# 从零开始构建推理引擎（一）：PyTorch 复现 Qwen3 推理过程

## 0. 准备虚拟环境

本文使用的 Python 版本为 3.11。读者可以使用自己青睐的虚拟环境工具，如 conda 或 virtualenv 或 uv 来创建本 notebook 所需的虚拟环境。

本文为读者提供了 `pyproject.toml` 描述虚拟环境所依赖的安装包，你可以使用 `uv sync` 或者其他方式来使用它。

## 1. 模型资源

### 1.1 下载模型资源

要实现某个模型的推理，首先需要获得模型的资源。
各个模型在开源的时候，一般都会在 huggingface 上发布自己的模型资源。
本文以 Qwen3-0.6B 为例，你可以在 [Qwen/Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B) 上找到模型资源。
模型资源一般使用 huggingface 提供的 [CLI](https://huggingface.co/docs/huggingface_hub/guides/cli) 来下载。

请首先按照 CLI 文档安装 `hf` 并下载 Qwen3-0.6B 的模型资源。
为了便于查看模型资源的内容，这次我们将模型资源下载到 `~/huggingface/Qwen3-0.6B/` 中。

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "true"  # 取消 huggingface 检测到进程 fork 的警告

In [ ]:
!hf download Qwen/Qwen3-0.6B --local-dir ~/huggingface/Qwen3-0.6B/

### 1.2 模型资源介绍

我们需要了解一下模型资源的构成，明确推理过程会用到哪些资源文件。

#### 1.2.1 模型配置 `config.json`

模型配置文件 `config.json` 中记录了模型参数，比如输入和输出的维度、隐藏层维度、层数等。
这些参数在推理中用于初始化模型。

In [ ]:
!cat ~/huggingface/Qwen3-0.6B/config.json

#### 1.2.2 模型权重 `model.safetensors`

模型权重张量，所有层的权重都保存在 `model.safetensors` 文件中。
该文件可以通过 `safetensors` 库打开，我们可粗浅看一下这些权重的结构。
打开后的内容是一个字典，键为层名称，值为该层的权重张量。

In [ ]:
import os
from safetensors import safe_open

path = os.path.expanduser("~/huggingface/Qwen3-0.6B/model.safetensors")

tensors = {}

with safe_open(path, framework="pt", device="cpu") as f:
    for key in f.keys()[:10]:
        tensors[key] = f.get_tensor(key)
        print(key, tensors[key].shape)

#### 1.2.3 分词器配置 `tokenizer.json`

模型使用的分词器配置，用于将输入文本转换为模型可处理的输入。
凭空举个例子，分词器将文本 `hello\w` 转换为 `hel` 和 `lo\w` 两个 token，分别映射到数字 123 和 987。

Qwen3 模型使用的分词是 Byte-Pair Encoding（BPE），是一种基于统计的方法，
将文本中的单词或字符进行编码，并生成一个编码表。
有兴趣的读者可以根据 BPE 算法和 `tokenizer.json` 自行实现一个分词器。

另外还有一个 `tokenizer_config.json` 文件，用于配置分词器的参数。
根据 `tokenizer_config.json` 可以看到 transformers 库所使用的分词器类是 `Qwen2Tokenizer`。

实际上也存在多种分词库。
这里我们简单使用 transformers 库中的分词器即可。
下面是分词的一个简单使用例子。

In [ ]:
import os
from transformers import AutoTokenizer

path = os.path.expanduser("~/huggingface/Qwen3-0.6B")

tokenizer = AutoTokenizer.from_pretrained(path)

# 用户原始的输入
prompt = "Hello, how are you?"
# 通过 chat_template 函数填充用户的输入，具体可以见 tokenizer_config.json 中对应的脚本
messages = [{"role": "user", "content": prompt}]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True,
)
print("-" * 40)
print(text)
print("-" * 40)
# 对输入进行编码，生成 token id 张量
# 形状为 [batch_size, seq_len]
# 这里我们复制一次输入，让 batch_size=2
model_inputs = tokenizer([text] * 2, return_tensors="pt")
for k in model_inputs.keys():
    print(k)
    print(model_inputs[k])
    print(model_inputs[k].shape)
# 我们还可以对这些 token id 张量进行解码
print("-" * 40)
print(tokenizer.decode(model_inputs.input_ids[0]))
print("-" * 40)

另外旧版本的分词器所使用的是 `merges.txt` 和 `vocab.json`，模型资源中会给出但不一定会用到。
这两个文件所包含的内容与 `tokenizer.json` 是等价的。

#### 1.2.4 生成配置 `generation_config.json`

最后是生成配置，用于控制生成过程。
其中记录了生成过程中使用的参数，如温度、TopK、TopP 等。
在编写生成 token 的逻辑时，会用到这些内容。

生成配置文件 `generation_config.json` 的内容如下：

In [ ]:
!cat ~/huggingface/Qwen3-0.6B/generation_config.json

## 2. 使用 Transformers 进行推理

对于我们来说，为了实现 Qwen3 模型进行推理，除了前面提到的模型资源以外，还需要准备模型推理代码。

模型在 huggingface 上发布的时候，已经提供了推理代码，有助于我们移植到任意其他推理引擎中。
要复现 Qwen3 的推理，我们实际上需要理解 Qwen3 的模型结构。
首先从输入输出复现一下我们要复现的模型是什么样的。
我们可以复制 huggingface 上 Qwen3 的官方示例并运行。

In [ ]:
import os
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "~/huggingface/Qwen3-0.6B"

path = os.path.expanduser(model_name)

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(path)
model = AutoModelForCausalLM.from_pretrained(
    path,
    torch_dtype="auto",
    device_map="auto",
)

# prepare the model input
prompt = "Give me a short introduction to large language model."
messages = [{"role": "user", "content": prompt}]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True,  # Switches between thinking and non-thinking modes. Default is True.
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# conduct text completion
generated_ids = model.generate(**model_inputs, max_new_tokens=32768)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]) :].tolist()

# parsing thinking content
try:
    # rindex finding 151668 (</think>)
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0

thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip(
    "\n"
)
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

print("thinking content:", thinking_content)
print("content:", content)

## 3. 根据 Transformers 源码实现 Qwen3 模型



接下来我们根据 